# <center> Advance SQL Techniques </center></center>

### The Complexity Crisis
In a real-world project, a single database is no longer just a storage bin for one application. It becomes a "hub" stressed by multiple roles:

* Analysts & Managers: Running massive, unoptimized SQL queries for reports.

* Data Engineers: Building ETL (Extract, Transform, Load) pipelines to move data into [Data Warehouses](collects data from different sources).

* Data Scientists: Pulling large datasets for Machine Learning models.

---

## Key Challenges Identified:
* `Redundancy`: Different people writing the same logic repeatedly, wasting effort.

* `Performance Bottlenecks`: Massive queries slowing down the system for everyone.

* `Model Complexity`: Users don't understand the underlying table relationships (the "Physical Model") and constantly bug developers for help.

* `Security Risks`: Giving everyone direct access to raw tables is dangerous.
---

**To solve these challenges, the text introduces five core SQL techniques designed to simplify logic, improve reuse, and boost performance:**

1. **Subqueries**

2. **CTEs** (Common Table Expressions)

3. **Views**

4. **Temporary Tables**

5. **CTAS** (Create Table As Select)
---
The Three Disk Storage Areas:
* `User Data:` The actual content you care about (e.g., Customers or Orders tables).

* `System Catalog (Metadata):` **Data about data** It acts as a blueprint, storing table structures, column types, and constraints. In SQL Server, this is accessed via the INFORMATION_SCHEMA.

* `Temporary Data:` A workspace (like the TempDB) used for short-term tasks like sorting or processing complex joins before the results are finalized.

---
**The Query Execution Flow**
When you run a query, the Database Engine follows a specific path:

A. It checks the Cache first (to see if the result is already available quickly).

B. If not in Cache, it reads from the Disk.

C. It uses Metadata to understand the table structure.

D. It may use Temporary Storage to process the math/sorting.

E. It returns the final result to your screen (the Client).

![a1](../Resources/assets/advance/a1.png)

---

# <center> 1. Sub Query </center>

- A subquery in SQL is a query nested inside another SQL query.  The query that contains a subquery is known as an `outer query/main query` and the inner one is also known as `embedded query`.


SQL queries use `SELECT`, `FROM`, and `WHERE` to retrieve data directly from database tables and return results to the user. With `subqueries`, this changes slightly:
1. SQL first executes the subquery. Subquery retrieves data from the database tables.
2. Its result is not shown to the user. Instead, it becomes an intermediate result.
3. The main query then uses this intermediate result for `filtering, joining, or other logic`, along with querying the original tables.
4. Finally, the main query returns the final result to the user. 

- The `intermediate result exists only during query execution and is destroyed afterward. 
- It is not accessible by other, separate SQL queries. 
- The subquery is visible only to its main query and servers as a supporting data source. 

### Why we need Sub Query

* Subqueries are used to manage complex SQL tasks by breaking them into smaller, logical steps.
* A complex query often involves multiple stages. Writing everything at once can lead to long, hard-to-read and hard-to-maintain queries. 
* Instead, each step can be written as a separate query, where: **Each step prepares data for the next one**

## How DB executes SubQueries

* When you run a query with a subquery, the database engine first identifies and executes the subquery. It retrieves the required data from disk and stores the intermediate result temporarily in cache/memory.
* After the subquery finishes, the database executes the main query, which uses the cached subquery result instead of rereading data from disk, making execution faster.
* Once the final result is produced and sent back to the client, the database cleans up the cache, removing the temporary subquery results to free space for other queries.

![sub4](../Resources/assets/advance/sub/s4.png)

### Types & Categories of Subqueries

#### 1. Based on Result Type Returned
![s1](../Resources/assets/advance/sub/s1.png)

- **Scalar Subquery**: Returns a single value
- **Row Subquery**: Returns multiple rows(single column)
- **Table Subquery**: Returns multiple rows and multiple columns. 

```sql
-- Find employees whose salary is greater than the average salary.
-- The subquery returns one value and that single value is used in a comparison.
SELECT employee_id, salary
FROM employees
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
);

-- Find employees who work in departments located in New York.
-- The subquery returns multipe department IDs and the main query checks if a value exists in that list
SELECT employee_id, department_id
FROM employees
WHERE department_id IN (
    SELECT department_id
    FROM departments
    WHERE location = 'New York'
);

--Get total salary per department, then filter departments with total salary > 50,000.
-- The subquery returns a full table (multiple rows and columns). The main query treats it like a temporary table.
SELECT department_id, total_salary
FROM (
    SELECT department_id, SUM(salary) AS total_salary
    FROM employees
    GROUP BY department_id
) dept_totals
WHERE total_salary > 50000;
```
#### 2. Based on Dependency with Main Query
![s2](../Resources/assets/advance/sub/s2.png)

- **Non-correlated Subquery**: Independent of the main query
- **Correlated Subquery**: Depends on the main query. `Executes in relation to each row of the main query`.
![s2](../Resources/assets/advance/sub/s6.png)

```sql
-- Independent of main query, can run on its own
-- Executed once, Result is reused by the main query
FROM employees
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
);

--Depends on the main query, Uses values from the main query
-- Executed once per row Cannot run independently
SELECT
    c.customer_id,
    c.customer_name,
    (
        SELECT COUNT(*)
        FROM sales_orders o
        WHERE o.customer_id = c.customer_id
    ) AS total_orders
FROM sales_customers c;
-- For each customer row:
-- The subquery runs only for that customer & Counts matching orders
```
#### 3. Based on Location and SQL Clause usage
![s3](../Resources/assets/advance/sub/s3.png)

Subqueries can be placed in 
- `SELECT`, `FROM`, `WHERE`, `JOINS` clauses

```sql
-- Show each employee’s salary and the company average salary
-- The subquery runs first and clculates the average salary. 
SELECT
    employee_id,
    salary,
    (SELECT AVG(salary) FROM employees) AS avg_salary
FROM employees;

-- Find departments iwth total salary above 50K
-- The subquery prepares aggregated data. The main query filters from that prepared dataset.

SELECT department_id, total_salary
FROM (
    SELECT department_id, SUM(salary) AS total_salary
    FROM employees
    GROUP BY department_id
) dept_totals
WHERE total_salary > 50000;

-- Employees earning more than the average salary
SELECT employee_id, salary
FROM employees
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
);
```

![s5](../Resources/assets/advance/sub/s5.png)

---

![s5](../Resources/assets/advance/sub/s7.png)

---

# <center> 2. CTE (Common Table Expressions)</center>

A `Common Table Expression (CTE)` is a temporary result set in SQL that you can reference within a single query. CTEs simplify complex queries, make them easier to read, and can be reused multiple times within the same query. It is used for:

- Split big queries into smaller, reusable pieces.
- **Virtual Table:** It functions like a table that lives only inside your query.
- **Temporary:** It exists only for the duration of the query execution. Once the query finishes, the CTE is destroyed (cleaned up).
- **Local Scope:** The CTE is available only to the "main query" that follows it; it is not globally available to other queries in your database.
